In [1]:
#Importing the NYC dataset that's focused on rodent complaints from 2021-2025
import pandas as pd
import requests #for api route

#Part a. Import
#Reading zip / BBL / additional_details as text: zips and BBLs are IDs, not numbers, and additional_details is ~97% empty, which causeses a mixed-type warning.
#CSV File
df = pd.read_csv(
    "queryed_dataset.csv",
    dtype={"Incident Zip": str, "Additional Details": str, "BBL": str},
    low_memory=False,
)

#Part b. Parse and Transform

#1. Renaming all the long column names, then standardizing all the names to snake_case
df = df.rename(columns={
    "Problem (formerly Complaint Type)": "Complaint Type",
    "Problem Detail (formerly Descriptor)": "Descriptor",
})
df.columns = (df.columns.str.strip().str.lower()
              .str.replace(r"[^\w]+", "_", regex=True).str.strip("_"))

#2. Parsing date strings ("12/31/2025 09:50:33 PM") into datetimes for consistency
date_fmt = "%m/%d/%Y %I:%M:%S %p"
for col in ["created_date", "closed_date", "resolution_action_updated_date"]:
    df[col] = pd.to_datetime(df[col], format=date_fmt, errors="coerce")

#3. Extracting time features into multiple columns
df["year"]        = df["created_date"].dt.year
df["month"]       = df["created_date"].dt.month
df["year_month"]  = df["created_date"].dt.to_period("M")
df["day_of_week"] = df["created_date"].dt.day_name()
df["hour"]        = df["created_date"].dt.hour
df["season"] = df["month"].map({12: "Winter", 1: "Winter", 2: "Winter",
                                3: "Spring", 4: "Spring", 5: "Spring",
                                6: "Summer", 7: "Summer", 8: "Summer",
                                9: "Fall", 10: "Fall", 11: "Fall"})

#4. New feature: days it took to resolve a rodent complaint
df["resolution_days"] = (df["closed_date"] - df["created_date"]).dt.total_seconds() / 86400

#5. Text-to-number conversions
for col in ["x_coordinate_state_plane", "y_coordinate_state_plane"]:   # "1,024,887" -> 1024887
    df[col] = pd.to_numeric(df[col].str.replace(",", ""), errors="coerce").astype("Int64")
df["police_precinct"] = df["police_precinct"].str.extract(r"(\d+)")[0].astype("Int64")  # "Precinct 112" -> 112
df["council_district"] = df["council_district"].astype("Int64")       # 29.0 -> 29 (nullable int)
df["community_board_num"] = df["community_board"].str.extract(r"^(\d+)")[0].astype("Int64")  # "06 QUEENS" -> 6
df["unique_key"] = df["unique_key"].astype(str)                       # an ID, not a quantity

#6. Feature extraction: turn the long resolution sentences into short outcome labels
def outcome(text):
    if pd.isna(text):
        return "No resolution recorded"
    t = text.lower()
    if "earlier complaint" in t:               return "Duplicate"
    if "minor violations" in t:                return "Passed (minor violations)"
    if "found violations" in t or "violations were identified" in t:
                                               return "Violations found"
    if "did not find any violations" in t:     return "Passed"
    if "could not access" in t or "unsafe" in t: return "No access"
    if "referred it to another agency" in t:   return "Referred"
    if "could not locate" in t:                return "Address not found"
    return "Other"

df["inspection_outcome"] = df["resolution_description"].map(outcome)

#7. Convert repeated text values to the category type (to save  memory, and have faster groupby processing)
cat_cols = ["descriptor", "additional_details", "location_type", "address_type", "city",
            "status", "borough", "open_data_channel_type", "day_of_week", "season",
            "inspection_outcome"]
for col in cat_cols:
    df[col] = df[col].astype("category")

print("Number of Complaints and Columns")
print(df.shape)
print(df.dtypes.to_string())


Number of Complaints and Columns
(192716, 53)
unique_key                                object
created_date                      datetime64[ns]
closed_date                       datetime64[ns]
agency                                    object
agency_name                               object
complaint_type                            object
descriptor                              category
additional_details                      category
location_type                           category
incident_zip                              object
incident_address                          object
street_name                               object
cross_street_1                            object
cross_street_2                            object
intersection_street_1                     object
intersection_street_2                     object
address_type                            category
city                                    category
landmark                                  object
facility_type          

#Part C: Organizing into Appropriate Data Structures

In [2]:
df.info() # There are a lot of empty columns that have empty values.

df = df.dropna(axis = 1, how = 'all') #drops all the columns with empty values.
df.columns


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 192716 entries, 0 to 192715
Data columns (total 53 columns):
 #   Column                          Non-Null Count   Dtype         
---  ------                          --------------   -----         
 0   unique_key                      192716 non-null  object        
 1   created_date                    192716 non-null  datetime64[ns]
 2   closed_date                     186874 non-null  datetime64[ns]
 3   agency                          192716 non-null  object        
 4   agency_name                     192716 non-null  object        
 5   complaint_type                  192716 non-null  object        
 6   descriptor                      192716 non-null  category      
 7   additional_details              6082 non-null    category      
 8   location_type                   192716 non-null  category      
 9   incident_zip                    192714 non-null  object        
 10  incident_address                192716 non-null  object 

Index(['unique_key', 'created_date', 'closed_date', 'agency', 'agency_name',
       'complaint_type', 'descriptor', 'additional_details', 'location_type',
       'incident_zip', 'incident_address', 'street_name', 'cross_street_1',
       'cross_street_2', 'intersection_street_1', 'intersection_street_2',
       'address_type', 'city', 'landmark', 'status', 'resolution_description',
       'resolution_action_updated_date', 'community_board', 'council_district',
       'police_precinct', 'bbl', 'borough', 'x_coordinate_state_plane',
       'y_coordinate_state_plane', 'open_data_channel_type',
       'park_facility_name', 'park_borough', 'latitude', 'longitude',
       'location', 'year', 'month', 'year_month', 'day_of_week', 'hour',
       'season', 'resolution_days', 'community_board_num',
       'inspection_outcome'],
      dtype='object')

In [3]:
# Setting unique_key as the Index
#df = df.set_index('unique_key')
df

,unique_key,created_date,closed_date,agency,agency_name,complaint_type,descriptor,additional_details,location_type,incident_zip,...,location,year,month,year_month,day_of_week,hour,season,resolution_days,community_board_num,inspection_outcome
0,67348969,2025-12-31 21:50:33,2026-01-09 13:44:00,DOHMH,Department of Health and Mental Hygiene,Rodent,Rat Sighting,NaN,3+ Family Mixed Use Building,11375,...,POINT (-73.853385496661 40.726848344855),2025,12,2025-12,Wednesday,21,Winter,8.662118,6,Violations found
1,67348967,2025-12-31 21:34:11,2026-01-06 12:12:00,DOHMH,Department of Health and Mental Hygiene,Rodent,Rat Sighting,NaN,3+ Family Apt. Building,11217,...,POINT (-73.972429604795 40.680307572764),2025,12,2025-12,Wednesday,21,Winter,5.609595,8,Violations found
2,67348968,2025-12-31 20:54:18,2026-01-07 10:54:31,DOHMH,Department of Health and Mental Hygiene,Rodent,Rat Sighting,NaN,3+ Family Mixed Use Building,10025,...,POINT (-73.962573996857 40.799657556389),2025,12,2025-12,Wednesday,20,Winter,6.583484,7,No access
3,67345388,2025-12-31 20:09:51,2026-01-05 11:25:14,DOHMH,Department of Health and Mental Hygiene,Rodent,Rat Sighting,NaN,3+ Family Apt. Building,10473,...,POINT (-73.856401581506 40.813973029887),2025,12,2025-12,Wednesday,20,Winter,4.635683,9,Violations found
4,67346580,2025-12-31 19:01:21,2025-12-31 19:01:21,DOHMH,Department of Health and Mental Hygiene,Rodent,Signs of Rodents,Holes or Burrows,3+ Family Apt. Building,11691,...,POINT (-73.748352764612 40.608141255885),2025,12,2025-12,Wednesday,19,Winter,0.000000,14,Duplicate
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
192711,48546584,2021-01-01 12:26:06,2021-01-19 11:01:43,DOHMH,Department of Health and Mental Hygiene,Rodent,Rat Sighting,NaN,3+ Family Apt. Building,10463,...,POINT (-73.90598546285 40.869870725786),2021,1,2021-01,Friday,12,Winter,17.941400,7,Duplicate
192712,48545367,2021-01-01 12:07:05,2021-01-13 13:34:04,DOHMH,Department of Health and Mental Hygiene,Rodent,Signs of Rodents,NaN,3+ Family Apt. Building,11374,...,POINT (-73.860538333127 40.731084159748),2021,1,2021-01,Friday,12,Winter,12.060405,6,Duplicate
192713,48546497,2021-01-01 12:05:11,2021-01-01 12:05:11,DOHMH,Department of Health and Mental Hygiene,Rodent,Rat Sighting,NaN,3+ Family Apt. Building,11374,...,POINT (-73.860538333127 40.731084159748),2021,1,2021-01,Friday,12,Winter,0.000000,6,Passed
192714,48545333,2021-01-01 09:54:42,2021-01-01 09:54:42,DOHMH,Department of Health and Mental Hygiene,Rodent,Condition Attracting Rodents,NaN,Other (Explain Below),11385,...,POINT (-73.903052919081 40.703872317877),2021,1,2021-01,Friday,9,Winter,0.000000,5,Passed


In [4]:
# Creating Lists of Column types
time_columns = ['created_date', 'closed_date', 'year', 'month', 'year_month',
                'day_of_week', 'hour', 'resolution_days',
                'season', 'resolution_action_updated_date']
complaint_columns = ['agency', 'agency_name', 'complaint_type','descriptor',
                     'additional_details', 'status', 'resolution_description',
                     'community_board', 'open_data_channel_type', 'inspection_outcome']
location_columns = ['location_type', 'incident_zip', 'incident_address', 'street_name',
                    'cross_street_1', 'cross_street_2', 'intersection_street_1',
                    'intersection_street_2', 'address_type', 'city', 'landmark',
                    'council_district', 'police_precinct', 'borough',
                    'x_coordinate_state_plane', 'y_coordinate_state_plane', 'location',
                    'park_facility_name', 'park_borough', 'latitude', 'longitude',
                    'community_board_num']
# Changing month_year into a string to become compatible with SQL
df['year_month'] = df['year_month'].astype(str)

In [5]:
# Setting up a SQL Database
import sqlite3
from sqlalchemy import create_engine
conn = sqlite3.connect(':memory:')
cursor = conn.cursor()
engine = create_engine('sqlite://', echo = False)


In [6]:
df[complaint_columns].to_sql('complaint', con = engine)
df[location_columns].to_sql('location', con = engine)
df[time_columns].to_sql('time', con = engine)

192716

#Part D: Cleaning Data